# Elution Sequence Prediction — Model Training

**Version: 1.4** | 2026-03-08 — Fix `total_mem` → `total_memory` for PyTorch 2.10+

**Purpose:** Train LSTM and Transformer models on the next-m/z-bin prediction task.  
**Runtime:** Open this notebook from Google Drive in Colab (GPU runtime).  
**Data:** Pre-tokenized features from 4 clinical lipidomics cohorts (342 samples, ~1.27M features).

**How to use:**
1. Open this notebook from Google Drive → "Open with" → Google Colab
2. Runtime → Change runtime type → T4 GPU
3. Run all cells sequentially
4. Results are saved back to the same Google Drive folder

**Monitoring:** Training progress is logged to `outputs/training_log.csv` on Google Drive after every epoch. Check this file from your local machine to monitor progress remotely.

**Changelog:**
- v1.4: Fix `total_mem` → `total_memory` AttributeError on PyTorch 2.10+ (Colab default)
- v1.3: Add persistent CSV training log to Drive (survives crashes). Includes timestamps, GPU memory, and status markers. Checkpoint saved after every epoch.
- v1.2: Add periodic checkpoint saving (every 10 epochs)
- v1.1: Switch working directory to `/content` after data load; set `num_workers=0`
- v1.0: Initial version

## 1. Setup & Data Loading

Mount Google Drive and set working directory to the notebook's folder so all paths are relative.

In [2]:
import subprocess, sys

# Colab already has torch with CUDA — just verify
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install pyarrow if needed (for parquet)
try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

import numpy as np
import pandas as pd

PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


In [ ]:
# Mount Google Drive, load data, then switch to local disk for training speed
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence/01_train_models"

tok = pd.read_parquet(f"{DRIVE_DIR}/tokenized_features.parquet")
print(f"Loaded: {len(tok):,} rows, {tok[['study','sample_id']].drop_duplicates().shape[0]} samples")
print(f"Columns: {list(tok.columns)}")

# Switch to local disk — avoids Drive I/O overhead during training
os.chdir("/content")
print(f"\nWorking directory set to /content (local disk) for training speed")

## 2. Configuration & Data Pipeline

All hyperparameters, tokenization constants, dataset splitting, and model code — self-contained in this notebook so no local file dependencies are needed on the Colab runtime.

In [ ]:
# === Configuration (mirrors config.yaml) ===
RANDOM_SEED = 42
CONTEXT_LENGTH = 64
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.1
NUM_HEADS = 4
FF_DIM = 256
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

RT_GAP_LABELS = ["co-elute", "0.1-0.5s", "0.5-1s", "1-2s", "2-5s", "5-15s", ">15s"]
INTENSITY_RANK_LABELS = ["top1%", "top5%", "top20%", "top50%", "low"]
POLARITY_MAP = {"pos": 0, "neg": 1, "unk": 2}
RT_GAP_MAP = {label: i for i, label in enumerate(RT_GAP_LABELS)}
INTENSITY_MAP = {label: i for i, label in enumerate(INTENSITY_RANK_LABELS)}

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# === Encode categorical fields ===
tok["polarity_idx"] = tok["polarity_tok"].map(POLARITY_MAP).fillna(2).astype(int)
tok["rt_gap_idx"] = tok["rt_gap_bin"].astype(str).map(RT_GAP_MAP).fillna(0).astype(int)
tok["intensity_idx"] = tok["intensity_rank"].astype(str).map(INTENSITY_MAP).fillna(4).astype(int)
tok = tok.sort_values(["study", "sample_id", "seq_pos"])

# === Sample-aware train/val/test split ===
rng = np.random.RandomState(RANDOM_SEED)
samples = tok[["study", "sample_id"]].drop_duplicates().reset_index(drop=True)

train_keys, val_keys, test_keys = set(), set(), set()
for study, group in samples.groupby("study"):
    indices = group.index.values.copy()
    rng.shuffle(indices)
    n = len(indices)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val = max(1, int(n * VAL_FRACTION))
    for idx in indices[:n_test]:
        test_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test:n_test + n_val]:
        val_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test + n_val:]:
        train_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))

def mask_for(keys):
    return tok.apply(lambda r: (r["study"], r["sample_id"]) in keys, axis=1)

train_df = tok[mask_for(train_keys)]
val_df = tok[mask_for(val_keys)]
test_df = tok[mask_for(test_keys)]

print(f"Train: {train_df[['study','sample_id']].drop_duplicates().shape[0]} samples, {len(train_df):,} rows")
print(f"Val:   {val_df[['study','sample_id']].drop_duplicates().shape[0]} samples, {len(val_df):,} rows")
print(f"Test:  {test_df[['study','sample_id']].drop_duplicates().shape[0]} samples, {len(test_df):,} rows")

In [ ]:
# === Build per-sample arrays and PyTorch Dataset ===
from torch.utils.data import Dataset, DataLoader

def build_sample_arrays(df):
    samples = []
    for (study, sid), group in df.groupby(["study", "sample_id"]):
        g = group.sort_values("seq_pos")
        samples.append({
            "study": study, "sample_id": sid,
            "mz_bin": g["mz_bin"].values.astype(np.int64),
            "md_bin": g["md_bin"].values.astype(np.int64),
            "rt_gap_idx": g["rt_gap_idx"].values.astype(np.int64),
            "polarity_idx": g["polarity_idx"].values.astype(np.int64),
            "intensity_idx": g["intensity_idx"].values.astype(np.int64),
            "rt_bin": g["rt_bin"].values.astype(np.int64),
        })
    return samples

class ElutionSequenceDataset(Dataset):
    def __init__(self, sample_arrays, context_length=CONTEXT_LENGTH):
        self.context_length = context_length
        self.sample_arrays = sample_arrays
        self.examples = []
        for si, s in enumerate(sample_arrays):
            for pos in range(context_length, len(s["mz_bin"])):
                self.examples.append((si, pos))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        si, pos = self.examples[idx]
        s = self.sample_arrays[si]
        start = pos - self.context_length
        return {
            "ctx_mz":  torch.tensor(s["mz_bin"][start:pos], dtype=torch.long),
            "ctx_md":  torch.tensor(s["md_bin"][start:pos], dtype=torch.long),
            "ctx_gap": torch.tensor(s["rt_gap_idx"][start:pos], dtype=torch.long),
            "ctx_pol": torch.tensor(s["polarity_idx"][start:pos], dtype=torch.long),
            "ctx_int": torch.tensor(s["intensity_idx"][start:pos], dtype=torch.long),
            "target_mz": torch.tensor(s["mz_bin"][pos], dtype=torch.long),
            "target_rt": torch.tensor(s["rt_bin"][pos], dtype=torch.long),
            "target_md": torch.tensor(s["md_bin"][pos], dtype=torch.long),
        }

train_arr = build_sample_arrays(train_df)
val_arr = build_sample_arrays(val_df)
test_arr = build_sample_arrays(test_df)

MAX_MZ_BIN = max(
    max(s["mz_bin"].max() for s in train_arr),
    max(s["mz_bin"].max() for s in val_arr),
    max(s["mz_bin"].max() for s in test_arr),
) + 1
print(f"Max m/z bin (num classes): {MAX_MZ_BIN}")

train_ds = ElutionSequenceDataset(train_arr)
val_ds = ElutionSequenceDataset(val_arr)
test_ds = ElutionSequenceDataset(test_arr)
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,} examples")

# num_workers=0 avoids subprocess overhead when data is already in RAM
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## 3. Model Definitions

In [ ]:
import torch.nn as nn
import math

class MultiFieldEmbedding(nn.Module):
    """Embed each token field separately, then sum."""
    def __init__(self, embedding_dim, max_mz_bin=120, max_md_bin=20,
                 max_rt_gap=7, max_polarity=3, max_intensity=5):
        super().__init__()
        self.mz_embed = nn.Embedding(max_mz_bin, embedding_dim)
        self.md_embed = nn.Embedding(max_md_bin, embedding_dim)
        self.gap_embed = nn.Embedding(max_rt_gap, embedding_dim)
        self.pol_embed = nn.Embedding(max_polarity, embedding_dim)
        self.int_embed = nn.Embedding(max_intensity, embedding_dim)

    def forward(self, mz, md, gap, pol, intensity):
        return (self.mz_embed(mz) + self.md_embed(md) + self.gap_embed(gap)
                + self.pol_embed(pol) + self.int_embed(intensity))


class LSTMModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.1, **embed_kw):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, num_mz_classes)

    def forward(self, mz, md, gap, pol, intensity):
        x = self.embedding(mz, md, gap, pol, intensity)
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :]))


class TransformerModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, num_heads=4,
                 ff_dim=256, num_layers=2, dropout=0.1,
                 context_length=64, **embed_kw):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kw)
        self.pos_embed = nn.Embedding(context_length, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(embedding_dim, num_mz_classes)
        self.register_buffer("causal_mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, mz, md, gap, pol, intensity):
        B, T = mz.shape
        x = self.embedding(mz, md, gap, pol, intensity)
        x = x + self.pos_embed(torch.arange(T, device=mz.device).unsqueeze(0))
        x = self.dropout(x)
        x = self.transformer(x, mask=self.causal_mask[:T, :T])
        return self.head(x[:, -1, :])


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Quick shape test
embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7, max_polarity=3, max_intensity=5)
lstm = LSTMModel(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **embed_kw)
tfm = TransformerModel(MAX_MZ_BIN, EMBEDDING_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS, DROPOUT, CONTEXT_LENGTH, **embed_kw)
print(f"LSTM:        {count_parameters(lstm):,} parameters")
print(f"Transformer: {count_parameters(tfm):,} parameters")

## 4. Training Loop

In [ ]:
import time, json, csv
from datetime import datetime
from torch.optim.lr_scheduler import CosineAnnealingLR

# Create output directory for checkpoints and logs
output_dir = f"{DRIVE_DIR}/outputs"
os.makedirs(output_dir, exist_ok=True)

LOG_PATH = f"{output_dir}/training_log.csv"
LOG_FIELDS = [
    "timestamp", "model", "epoch", "max_epochs", "status",
    "train_loss", "train_acc", "val_loss", "val_top1", "val_top5", "val_mae_da",
    "best_epoch", "best_val_loss", "lr", "epoch_time_s", "total_time_s",
    "gpu_mem_used_mb", "gpu_mem_total_mb",
]


def log_to_csv(row_dict):
    """Append one row to the persistent training log on Drive."""
    file_exists = os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def get_gpu_memory():
    """Return (used_mb, total_mb) for current GPU, or (0, 0) if no GPU."""
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e6
        total = torch.cuda.get_device_properties(0).total_memory / 1e6
        return round(used, 1), round(total, 1)
    return 0, 0


def evaluate(model, loader, criterion, top_k=(1, 3, 5, 10)):
    model.eval()
    total_loss, total_n = 0, 0
    total_correct = {k: 0 for k in top_k}
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            mz = batch["ctx_mz"].to(device)
            md = batch["ctx_md"].to(device)
            gap = batch["ctx_gap"].to(device)
            pol = batch["ctx_pol"].to(device)
            inten = batch["ctx_int"].to(device)
            target = batch["target_mz"].to(device)
            logits = model(mz, md, gap, pol, inten)
            loss = criterion(logits, target)
            total_loss += loss.item() * target.size(0)
            for k in top_k:
                _, topk_preds = logits.topk(k, dim=1)
                total_correct[k] += topk_preds.eq(target.unsqueeze(1)).any(dim=1).sum().item()
            pred_mz = logits.argmax(dim=1)
            all_preds.append(pred_mz.cpu().numpy())
            all_targets.append(target.cpu().numpy())
            total_n += target.size(0)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    mae_bins = np.mean(np.abs(all_preds.astype(float) - all_targets.astype(float)))
    metrics = {"loss": total_loss / total_n, "mae_da": float(mae_bins * 10), "n": total_n}
    for k in top_k:
        metrics[f"top{k}"] = total_correct[k] / total_n
    return metrics


def train_model(model, model_name, max_epochs=MAX_EPOCHS, patience=PATIENCE, lr=LEARNING_RATE):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs)

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None
    history = []
    training_start = time.time()

    # Config dict for checkpoint
    if model_name == "LSTM":
        config = {"embedding_dim": EMBEDDING_DIM, "hidden_dim": HIDDEN_DIM,
                  "num_layers": NUM_LAYERS, "num_classes": MAX_MZ_BIN}
    else:
        config = {"embedding_dim": EMBEDDING_DIM, "num_heads": NUM_HEADS,
                  "ff_dim": FF_DIM, "num_layers": NUM_LAYERS,
                  "context_length": CONTEXT_LENGTH, "num_classes": MAX_MZ_BIN}

    ckpt_path = f"{output_dir}/{model_name.lower()}_best.pt"

    print(f"\n{'='*70}")
    print(f"Training {model_name} ({count_parameters(model):,} params) on {device}")
    print(f"Checkpoint + log saved to Drive after EVERY epoch")
    print(f"Log: {LOG_PATH}")
    print(f"{'='*70}")

    # Log training start
    gpu_used, gpu_total = get_gpu_memory()
    log_to_csv({
        "timestamp": datetime.now().isoformat(), "model": model_name,
        "epoch": 0, "max_epochs": max_epochs, "status": "STARTED",
        "train_loss": "", "train_acc": "", "val_loss": "", "val_top1": "",
        "val_top5": "", "val_mae_da": "", "best_epoch": "", "best_val_loss": "",
        "lr": lr, "epoch_time_s": "", "total_time_s": 0,
        "gpu_mem_used_mb": gpu_used, "gpu_mem_total_mb": gpu_total,
    })

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss, train_correct, train_n = 0, 0, 0
        t0 = time.time()
        for batch in train_loader:
            mz = batch["ctx_mz"].to(device)
            md = batch["ctx_md"].to(device)
            gap = batch["ctx_gap"].to(device)
            pol = batch["ctx_pol"].to(device)
            inten = batch["ctx_int"].to(device)
            target = batch["target_mz"].to(device)
            optimizer.zero_grad()
            logits = model(mz, md, gap, pol, inten)
            loss = criterion(logits, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * target.size(0)
            train_correct += (logits.argmax(1) == target).sum().item()
            train_n += target.size(0)
        scheduler.step()
        dt = time.time() - t0
        total_time = time.time() - training_start

        val_m = evaluate(model, val_loader, criterion)
        rec = {
            "epoch": epoch, "train_loss": train_loss/train_n,
            "train_acc": train_correct/train_n,
            "val_loss": val_m["loss"], "val_top1": val_m["top1"],
            "val_top5": val_m["top5"], "val_mae_da": val_m["mae_da"],
            "lr": optimizer.param_groups[0]["lr"], "time_s": dt,
        }
        history.append(rec)
        print(f"Epoch {epoch:3d}/{max_epochs} | "
              f"loss={rec['train_loss']:.4f} acc={rec['train_acc']:.4f} | "
              f"val_loss={rec['val_loss']:.4f} top1={rec['val_top1']:.4f} "
              f"top5={rec['val_top5']:.4f} MAE={rec['val_mae_da']:.0f}Da | {dt:.0f}s")

        improved = val_m["loss"] < best_val_loss
        if improved:
            best_val_loss = val_m["loss"]
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # === Save checkpoint + log after EVERY epoch ===
        if best_state is not None:
            torch.save({
                "model_state_dict": best_state,
                "config": config,
                "history": history,
                "best_epoch": best_epoch,
            }, ckpt_path)

        gpu_used, gpu_total = get_gpu_memory()
        log_to_csv({
            "timestamp": datetime.now().isoformat(), "model": model_name,
            "epoch": epoch, "max_epochs": max_epochs,
            "status": "IMPROVED" if improved else "PLATEAU",
            "train_loss": f"{rec['train_loss']:.6f}",
            "train_acc": f"{rec['train_acc']:.6f}",
            "val_loss": f"{rec['val_loss']:.6f}",
            "val_top1": f"{rec['val_top1']:.6f}",
            "val_top5": f"{rec['val_top5']:.6f}",
            "val_mae_da": f"{rec['val_mae_da']:.1f}",
            "best_epoch": best_epoch,
            "best_val_loss": f"{best_val_loss:.6f}",
            "lr": f"{rec['lr']:.8f}",
            "epoch_time_s": f"{dt:.1f}",
            "total_time_s": f"{total_time:.1f}",
            "gpu_mem_used_mb": gpu_used,
            "gpu_mem_total_mb": gpu_total,
        })

        if epoch - best_epoch >= patience:
            print(f"\nEarly stopping at epoch {epoch} (best: {best_epoch})")
            log_to_csv({
                "timestamp": datetime.now().isoformat(), "model": model_name,
                "epoch": epoch, "max_epochs": max_epochs, "status": "EARLY_STOPPED",
                "train_loss": "", "train_acc": "", "val_loss": "", "val_top1": "",
                "val_top5": "", "val_mae_da": "", "best_epoch": best_epoch,
                "best_val_loss": f"{best_val_loss:.6f}", "lr": "",
                "epoch_time_s": "", "total_time_s": f"{total_time:.1f}",
                "gpu_mem_used_mb": gpu_used, "gpu_mem_total_mb": gpu_total,
            })
            break

    # Load best and evaluate on test
    model.load_state_dict(best_state)
    model = model.to(device)
    test_m = evaluate(model, test_loader, criterion)
    print(f"\n{'='*60}")
    print(f"TEST RESULTS — {model_name} (best epoch {best_epoch})")
    print(f"{'='*60}")
    print(f"  Loss:    {test_m['loss']:.4f}")
    print(f"  Top-1:   {test_m['top1']:.4f}")
    print(f"  Top-3:   {test_m['top3']:.4f}")
    print(f"  Top-5:   {test_m['top5']:.4f}")
    print(f"  Top-10:  {test_m['top10']:.4f}")
    print(f"  MAE:     {test_m['mae_da']:.0f} Da")

    # Save final checkpoint with test metrics
    torch.save({
        "model_state_dict": best_state,
        "test_metrics": test_m,
        "config": config,
        "history": history,
        "best_epoch": best_epoch,
    }, ckpt_path)
    print(f"Final checkpoint saved: {ckpt_path}")

    # Log completion
    log_to_csv({
        "timestamp": datetime.now().isoformat(), "model": model_name,
        "epoch": best_epoch, "max_epochs": max_epochs, "status": "COMPLETED",
        "train_loss": "", "train_acc": "",
        "val_loss": f"{test_m['loss']:.6f}",
        "val_top1": f"{test_m['top1']:.6f}",
        "val_top5": f"{test_m['top5']:.6f}",
        "val_mae_da": f"{test_m['mae_da']:.1f}",
        "best_epoch": best_epoch,
        "best_val_loss": f"{best_val_loss:.6f}", "lr": "",
        "epoch_time_s": "", "total_time_s": f"{time.time() - training_start:.1f}",
        "gpu_mem_used_mb": gpu_used, "gpu_mem_total_mb": gpu_total,
    })

    return model, test_m, history

## 5. Train LSTM

In [ ]:
embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7, max_polarity=3, max_intensity=5)

lstm_model = LSTMModel(MAX_MZ_BIN, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, **embed_kw)
lstm_model, lstm_test, lstm_history = train_model(lstm_model, "LSTM")

## 6. Train Transformer

In [ ]:
tfm_model = TransformerModel(MAX_MZ_BIN, EMBEDDING_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS, DROPOUT, CONTEXT_LENGTH, **embed_kw)
tfm_model, tfm_test, tfm_history = train_model(tfm_model, "Transformer")

## 7. Compare Results & Save

In [ ]:
import json

# Baseline results for comparison
baselines = {
    "Random":              {"top1": 0.0109, "top5": None, "mae_da": 1965},
    "Global frequency":    {"top1": 0.0340, "top5": None, "mae_da": None},
    "Same-as-previous":    {"top1": 0.2314, "top5": None, "mae_da": 1190},
    "Markov-1 (m/z)":      {"top1": 0.2510, "top5": 0.5827, "mae_da": 1150},
    "Joint (RT,m/z) Markov":{"top1": 0.5680, "top5": None, "mae_da": 670},
}

print(f"{'='*75}")
print(f"{'Model':<28} {'Top-1':<8} {'Top-3':<8} {'Top-5':<8} {'Top-10':<8} {'MAE(Da)':<8}")
print(f"{'-'*75}")
for name, m in baselines.items():
    t1 = f"{m['top1']:.4f}" if m['top1'] else "—"
    t5 = f"{m['top5']:.4f}" if m.get('top5') else "—"
    mae = f"{m['mae_da']:.0f}" if m.get('mae_da') else "—"
    print(f"  {name:<26} {t1:<8} {'—':<8} {t5:<8} {'—':<8} {mae:<8}")
print(f"{'-'*75}")
for name, m in [("LSTM", lstm_test), ("Transformer", tfm_test)]:
    print(f"  {name:<26} {m['top1']:.4f}  {m['top3']:.4f}  {m['top5']:.4f}  {m['top10']:.4f}  {m['mae_da']:.0f}")
print(f"{'='*75}")

# Save all results as JSON for download
results = {
    "lstm": {"test_metrics": lstm_test, "history": lstm_history,
             "params": count_parameters(lstm_model)},
    "transformer": {"test_metrics": tfm_test, "history": tfm_history,
                    "params": count_parameters(tfm_model)},
}
with open("neural_model_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\
Saved: neural_model_results.json")

In [ ]:
# === Training curves ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for name, hist, color in [("LSTM", lstm_history, "tab:blue"), ("Transformer", tfm_history, "tab:orange")]:
    epochs = [h["epoch"] for h in hist]
    axes[0].plot(epochs, [h["val_loss"] for h in hist], label=name, color=color)
    axes[1].plot(epochs, [h["val_top1"] for h in hist], label=name, color=color)
    axes[2].plot(epochs, [h["val_top5"] for h in hist], label=name, color=color)

axes[0].set_ylabel("Validation Loss"); axes[0].set_xlabel("Epoch")
axes[1].set_ylabel("Val Top-1 Accuracy"); axes[1].set_xlabel("Epoch")
axes[2].set_ylabel("Val Top-5 Accuracy"); axes[2].set_xlabel("Epoch")

# Add baseline references
axes[1].axhline(0.568, ls="--", color="gray", alpha=0.7, label="Joint Markov baseline")
axes[2].axhline(0.583, ls="--", color="gray", alpha=0.7, label="Markov-1 top-5")

for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle("Elution Sequence Prediction — Training Curves", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

## 8. Save Checkpoints & Download Results

Download the results JSON and training curves back to your local machine.

In [ ]:
# Checkpoints are already saved by train_model() — just save results JSON and training curves here

# Results JSON
with open(f"{output_dir}/neural_model_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Training curves figure
plt.savefig(f"{output_dir}/training_curves.png", dpi=150, bbox_inches="tight")

print(f"All outputs saved to {output_dir}")
print("Files:")
for f in os.listdir(output_dir):
    size = os.path.getsize(f"{output_dir}/{f}")
    print(f"  {f:<40} {size/1024:.0f} KB")